# Merge City Listings for ML
**Inputs:** `../Data/interim/{city}_listings_clean.parquet` (Madrid · Barcelona · Málaga)  
**Output:** `../Data/processed/listings_all_cities.parquet`  
**Prerequisite:** run each city's cleaning notebook first.

Concatenates the three cleaned listing files, adds a `city` column as the first column,
runs basic validation, and saves a single parquet ready for ML feature engineering.

In [ ]:
import pathlib
import numpy as np
import pandas as pd

CITY_ORDER = ['Madrid', 'Barcelona', 'Málaga']

_PATHS = {
    'Madrid':    '../Data/interim/madrid_listings_clean.parquet',
    'Barcelona': '../Data/interim/barcelona_listings_clean.parquet',
    'Málaga':    '../Data/interim/malaga_listings_clean.parquet',
}

OUTPUT_PATH = pathlib.Path('../Data/processed/listings_all_cities.parquet')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Load and tag each city

In [2]:
frames = []
for city in CITY_ORDER:
    path = pathlib.Path(_PATHS[city])
    if not path.exists():
        print(f'[WARNING] {city}: file not found — {path}')
        continue
    df = pd.read_parquet(path)
    if df.empty:
        print(f'[WARNING] {city}: parquet is empty')
        continue
    df.insert(0, 'city', city)
    frames.append(df)
    print(f'{city:12s}  {len(df):>6,} listings | {df.shape[1]} cols')

if not frames:
    raise RuntimeError('No city data loaded — check parquet paths above.')

Madrid        18,862 listings | 80 cols
Barcelona     15,199 listings | 80 cols
Málaga         8,762 listings | 80 cols


## 2. Concatenate

In [3]:
all_df = pd.concat(frames, join='outer', ignore_index=True)
all_df['city'] = pd.Categorical(all_df['city'], categories=CITY_ORDER, ordered=True)

print(f'Combined: {all_df.shape[0]:,} rows × {all_df.shape[1]} cols')
all_df['city'].value_counts().reindex(CITY_ORDER)

Combined: 42,823 rows × 80 cols


city
Madrid       18862
Barcelona    15199
Málaga        8762
Name: count, dtype: int64

## 3. Sanity checks

In [4]:
# Duplicate listing IDs across cities are expected (same id space per InsideAirbnb source)
# but within a city they should be unique
dup_within_city = (
    all_df.groupby('city')['id']
    .apply(lambda s: s.duplicated().sum())
)
print('Duplicate IDs within city:')
print(dup_within_city.to_string())

Duplicate IDs within city:
city
Madrid       0
Barcelona    0
Málaga       0


/var/folders/lp/8rxwgj595hs7vsc7ckn2fm840000gn/T/ipykernel_27382/4254949068.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  all_df.groupby('city')['id']


In [5]:
# Missing value summary — columns with >5% nulls
null_pct = all_df.isnull().mean().mul(100).round(2)
high_null = null_pct[null_pct > 5].sort_values(ascending=False)
print(f'Columns with >5% nulls ({len(high_null)}/{all_df.shape[1]}):')
print(high_null.to_string() if not high_null.empty else 'None')

Columns with >5% nulls (11/80):
neighbourhood_group_cleansed    20.46
review_scores_accuracy          17.28
review_scores_cleanliness       17.28
review_scores_checkin           17.28
review_scores_rating            17.27
review_scores_communication     17.27
review_scores_location          17.27
review_scores_value             17.27
days_since_first_review         17.27
days_since_last_review          17.27
review_span_years               17.27


In [6]:
# Column dtype overview
dtype_summary = all_df.dtypes.astype(str).value_counts().rename('count')
print('Dtype distribution:')
print(dtype_summary.to_string())

Dtype distribution:
float64           25
int64             23
object            18
datetime64[ns]     5
bool               5
category           4


In [7]:
# Key ML target and feature range check
for col in ['price', 'estimated_revenue_l365d', 'estimated_occupancy_l365d', 'review_scores_rating']:
    if col not in all_df.columns:
        continue
    print(f'\n{col}:')
    print(all_df.groupby('city')[col].describe().round(2).to_string())


price:
             count    mean     std   min   25%    50%    75%     max
city                                                                
Madrid     18862.0  135.62  112.22   8.0  70.0  110.0  162.0  1000.0
Barcelona  15199.0  168.32  162.65   9.0  69.0  130.0  213.0  1849.0
Málaga      8762.0  225.03  843.99  16.0  76.0  102.0  145.0  9429.0

estimated_revenue_l365d:
             count      mean       std  min    25%     50%      75%        max
city                                                                          
Madrid     18862.0  13021.52  17824.11  0.0  462.0  6474.0  19635.0   254490.0
Barcelona  15199.0  16234.78  23681.46  0.0    0.0  6288.0  23412.0   471495.0
Málaga      8762.0   9433.12  22910.49  0.0  750.0  4113.0  12636.0  1042302.0

estimated_occupancy_l365d:
             count    mean    std  min  25%   50%    75%    max
city                                                           
Madrid     18862.0  101.20  99.47  0.0  6.0  64.0  204.0  255.0
Barcel

/var/folders/lp/8rxwgj595hs7vsc7ckn2fm840000gn/T/ipykernel_27382/1190270373.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(all_df.groupby('city')[col].describe().round(2).to_string())
/var/folders/lp/8rxwgj595hs7vsc7ckn2fm840000gn/T/ipykernel_27382/1190270373.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(all_df.groupby('city')[col].describe().round(2).to_string())
/var/folders/lp/8rxwgj595hs7vsc7ckn2fm840000gn/T/ipykernel_27382/1190270373.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to ret

## 4. Save

In [8]:
all_df.to_parquet(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'Saved → {OUTPUT_PATH}  ({size_mb:.1f} MB)')
print(f'Shape : {all_df.shape[0]:,} rows × {all_df.shape[1]} cols')
print(f'Cols  : city is first — {list(all_df.columns[:6])} ...')

Saved → ../Data/processed/listings_all_cities.parquet  (23.9 MB)
Shape : 42,823 rows × 80 cols
Cols  : city is first — ['city', 'id', 'scrape_id', 'last_scraped', 'source', 'name'] ...
